# Levantamiento de observaciones del revisor — MapBiomas Perú
## Notebook complementario al artículo principal

Este notebook resuelve las **2 observaciones mayores** y las **5 menores** identificadas en la revisión.
Ejecutar en orden. Requiere que `tabla_maestra_final.csv`, `alertas_con_distancia.shp/gpkg` y el modelo `rf` estén disponibles en Drive.


---
## 0. Setup y carga de datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score,
                             classification_report, confusion_matrix,
                             brier_score_loss, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve
from scipy.stats import chi2_contingency, fisher_exact
import joblib

RUTA      = "/content/drive/MyDrive/Proyecto_MapBiomas/"
RUTA_DATA = RUTA + "data_limpia/"
RUTA_FIGS = RUTA + "figuras_modelo/"
import os; os.makedirs(RUTA_FIGS, exist_ok=True)

# ── Cargar datos maestros ─────────────────────────────────────
maestra = pd.read_csv(RUTA_DATA + "tabla_maestra_final.csv")
print(f"Registros totales en maestra: {len(maestra):,}")
print(f"Columnas: {maestra.columns.tolist()}")


---
## OBSERVACIÓN MAYOR 1 — Documentación del pipeline de filtrado
### "El texto no describe cómo pasar de las alertas crudas de MapBiomas a las 21,870 alertas finales"

Esta celda reconstruye el **funnel de decisión** completo y lo exporta como tabla y figura para la sección de Métodos.


In [ ]:
# ══════════════════════════════════════════════════════════════
# FUNNEL DE FILTRADO — de alertas crudas a dataset final
# Reproduce los pasos aplicados en GEE + Python para MDPI
# ══════════════════════════════════════════════════════════════

# Paso 1: alertas brutas descargadas de MapBiomas Alerta Perú
#   → Solo 9 departamentos amazónicos, área ≥ 0.5 ha
n_raw        = len(maestra)                          # total antes de filtros de calidad
n_con_geom   = maestra['flag_excluir'].notna().sum() # con geometría válida

# Paso 2: Filtro T-1 — excluir si cobertura previa no era bosque
n_flag_excluir = (maestra['flag_excluir'] == 1).sum()
n_flag_sinT1p  = (maestra.get('flag_sin_T1p', pd.Series(0, index=maestra.index)) == 1).sum()

# Paso 3: alertas con seguimiento disponible (2021-2023)
df_conT1 = maestra[maestra['flag_excluir'] == 0].copy()
n_conT1  = len(df_conT1)

df_final = df_conT1[
    df_conT1['ani_lrt'].isin([2021, 2022, 2023]) &
    df_conT1.get('flag_sin_T1p', pd.Series(0, index=df_conT1.index)).eq(0)
].copy()
n_final  = len(df_final)

# ── Tabla del funnel ────────────────────────────────────────────
funnel = pd.DataFrame({
    'Paso': [
        '0. Alertas descargadas (2021–2025, ≥0.5 ha, 9 dptos)',
        '1. Excluidas: T−1 no boscoso (clase_T1 = "otros")',
        '2. Excluidas: alertas 2024–2025 (sin T+1 en Colección 3.0)',
        '3. Excluidas: sin T+1 disponible (flag_sin_T1p)',
        '4. Dataset final para análisis (2021–2023)',
    ],
    'N': [
        n_raw,
        n_flag_excluir,
        n_raw - n_flag_excluir - n_flag_sinT1p - n_final,
        n_flag_sinT1p,
        n_final,
    ],
    'Tipo': ['Total', 'Excluido', 'Reservado', 'Excluido', 'Incluido']
})

funnel['% del total'] = (funnel['N'] / n_raw * 100).round(1)
print("=== FUNNEL DE FILTRADO ===")
print(funnel.to_string(index=False))

# ── Exportar CSV para la sección de métodos ─────────────────────
funnel.to_csv(RUTA_DATA + "pipeline_funnel_filtrado.csv", index=False)
print("\n✅ pipeline_funnel_filtrado.csv guardado")

# ── Figura del funnel ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
colores = {'Total': '#1A3A5C', 'Excluido': '#C0392B',
           'Reservado': '#E67E22', 'Incluido': '#1A5C2A'}
barras = [n_raw, n_raw - n_flag_excluir,
          n_raw - n_flag_excluir - n_flag_sinT1p, n_final]
etiquetas = [
    f"Alertas crudas\n(n = {n_raw:,})",
    f"Filtro T−1 boscoso\n(−{n_flag_excluir:,})",
    f"Solo 2021–2023\n(2024–2025 reservados)",
    f"Dataset final\n(n = {n_final:,})",
]
cols = ['#1A3A5C', '#3498DB', '#27AE60', '#1A5C2A']

ax.barh(range(4), barras[::-1], color=cols[::-1], edgecolor='white', height=0.55)
ax.set_yticks(range(4))
ax.set_yticklabels(etiquetas[::-1], fontsize=9)
ax.set_xlabel("Número de alertas", fontsize=10)
ax.set_title("Pipeline de filtrado: de alertas crudas al dataset analítico\n(MapBiomas Alerta Perú 2021–2025)",
             fontsize=11, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
for i, v in enumerate(barras[::-1]):
    ax.text(v + 50, i, f"{v:,}", va='center', fontsize=9)
plt.tight_layout()
plt.savefig(RUTA_FIGS + "obs_mayor1_pipeline_funnel.png", dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura funnel guardada")


In [ ]:
# ── Estadísticas adicionales sobre la máscara de pseudo-alertas ──
# Reproduce lo que hace el script GEE de pseudo-alertas para documentarlo

print("=== DOCUMENTACIÓN DE LA MÁSCARA DE PSEUDO-ALERTAS ===")
print()
print("Criterios aplicados en GEE (script pseudo_alertas_control.js):")
print("  1. Cobertura 2020: bosque primario (clase 3) O formación natural (clase 4)")
print("  2. Sin ninguna alerta MapBiomas verificada en el perímetro 2021–2023")
print("  3. Muestreo aleatorio estratificado a escala 500 m")
print("  4. Semilla aleatoria: 42 (reproducible)")
print("  5. N solicitados: 2,000 | N obtenidos tras máscara: 1,273")
print()
print("Tasa de rechazo de la máscara: {:.1f}%".format((1 - 1273/2000)*100))
print("Causa: bosques con overlap con alertas excluidos para garantizar")
print("       que el grupo control está libre de perturbación detectada.")
print()

# Si tienes el CSV de pseudo-alertas, cargarlo y mostrar distribución
try:
    pseudo = pd.read_csv(RUTA + "pseudo_alertas_control.csv")
    print(f"Pseudo-alertas cargadas: {len(pseudo):,}")
    print("\nDistribución T+1 (control):")
    print(pseudo['destino_T1'].value_counts().to_string())
    print(f"\nIESA T+1 (control): {(pseudo['iesa_T1']==1).mean()*100:.1f}%")
    print(f"IESA T+2 (control): {(pseudo['iesa_T2']==1).mean()*100:.1f}%")
    print(f"IESA T+3 (control): {(pseudo['iesa_T3']==1).mean()*100:.1f}%")
except FileNotFoundError:
    print("⚠ pseudo_alertas_control.csv no encontrado en Drive.")
    print("  Ejecutar primero: script_GEE/pseudo_alertas_control.js")


---
## OBSERVACIÓN MAYOR 2 — Caída del AUC espacial (0.887 → 0.746)
### "No discute qué features son responsables del overfitting geográfico ni ofrece estrategias de mejora"

Análisis completo: (a) SHAP por dominio geográfico, (b) importancia diferencial entrenamiento vs. Loreto+MDD, (c) estrategias de mejora.


In [ ]:
# ══════════════════════════════════════════════════════════════
# PREPARAR SPLITS PARA VALIDACIÓN ESPACIAL
# ══════════════════════════════════════════════════════════════

CONTINUAS  = ['log_are', 'log_dist_via', 'gini', 'ani_lrt']
CATEGORICAS = ['anp_categoria', 'ti_final', 'tip_lrt', 'dprtmnt', 'clase_T1']
BINARIAS   = ['carretera_cerca', 'dentro_ANP_dummy', 'flag_mixta',
              'interaccion_carretera_anp']

df_modelo = maestra[maestra['flag_excluir'] == 0].copy()
df_modelo = df_modelo[df_modelo['ani_lrt'].isin([2021, 2022, 2023])].copy()
df_modelo = df_modelo[df_modelo.get('flag_sin_T1p',
                pd.Series(0, index=df_modelo.index)).eq(0)].copy()

# Renombrar si necesario
if 'log_are' not in df_modelo.columns and 'log_area' in df_modelo.columns:
    df_modelo = df_modelo.rename(columns={'log_area': 'log_are'})
if 'log_dist_via' not in df_modelo.columns and 'log_dist_v' in df_modelo.columns:
    df_modelo = df_modelo.rename(columns={'log_dist_v': 'log_dist_via'})

FEATS_DISP = [f for f in CONTINUAS + CATEGORICAS + BINARIAS if f in df_modelo.columns]
df_encoded = pd.get_dummies(df_modelo[FEATS_DISP], columns=[c for c in CATEGORICAS if c in df_modelo.columns])
X_all = df_encoded.astype(float)
y_all = df_modelo['destino']

# Split espacial: entrenar en 7 dptos, evaluar en Loreto + Madre de Dios
DPTOS_EVAL = ['Loreto', 'Madre de Dios']
# ajustar mayúsculas/minúsculas según el dataset
dprtmnt_col = df_modelo['dprtmnt'].str.lower()
mask_eval   = dprtmnt_col.isin([d.lower() for d in DPTOS_EVAL])
mask_train  = ~mask_eval

X_train_sp = X_all[mask_train]
X_test_sp  = X_all[mask_eval]
y_train_sp = y_all[mask_train]
y_test_sp  = y_all[mask_eval]

print(f"Entrenamiento (7 dptos): {len(X_train_sp):,} alertas")
print(f"Evaluación (Loreto + MDD): {len(X_test_sp):,} alertas")
print(f"Distribución en evaluación:\n{y_test_sp.value_counts().to_string()}")


In [ ]:
# ── Entrenar RF para validación espacial ─────────────────────
rf_espacial = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_espacial.fit(X_train_sp, y_train_sp)

# Métricas globales
y_pred_sp   = rf_espacial.predict(X_test_sp)
y_proba_sp  = rf_espacial.predict_proba(X_test_sp)
clases_sp   = rf_espacial.classes_

y_bin_sp = label_binarize(y_test_sp, classes=clases_sp)
auc_macro_sp = roc_auc_score(y_bin_sp, y_proba_sp, average='macro', multi_class='ovr')
bal_acc_sp   = balanced_accuracy_score(y_test_sp, y_pred_sp)

print(f"AUC macro (validación espacial): {auc_macro_sp:.3f}")
print(f"Balanced accuracy (validación espacial): {bal_acc_sp:.3f}")
print()
print(classification_report(y_test_sp, y_pred_sp, digits=3))

# AUC por clase
for i, clase in enumerate(clases_sp):
    auc_c = roc_auc_score(y_bin_sp[:, i], y_proba_sp[:, i])
    print(f"  AUC {clase}: {auc_c:.3f}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# ANÁLISIS DE OVERFITTING GEOGRÁFICO
# ¿Qué features son responsables de la caída 0.887 → 0.746?
# Estrategia: comparar importancias RF entrenado vs. dominio espacial
# ══════════════════════════════════════════════════════════════

# RF aleatorio (referencia)
rf_aleatorio = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
X_tr_al, X_te_al, y_tr_al, y_te_al = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all)
rf_aleatorio.fit(X_tr_al, y_tr_al)

# Comparar importancias: RF aleatorio vs. RF espacial (entrenado en 7 dptos)
imp_aleatorio = pd.Series(rf_aleatorio.feature_importances_, index=X_all.columns)
imp_espacial  = pd.Series(rf_espacial.feature_importances_,  index=X_train_sp.columns)

# Diferencia (positivo = feature más importante en aleatorio que en espacial → señal de overfitting)
imp_diff = (imp_aleatorio - imp_espacial).sort_values(ascending=False)

print("=== FEATURES CON MAYOR DELTA (aleatorio − espacial) ===")
print("Positivo → el split aleatorio infla la importancia (posible overfitting geográfico)")
print()
print(imp_diff.head(15).to_string())
print()
print("=== FEATURES MÁS IMPORTANTES EN DOMINIO ESPACIAL (generalizables) ===")
print(imp_espacial.sort_values(ascending=False).head(10).to_string())

# ── Figura comparativa ──────────────────────────────────────
top_feat = imp_aleatorio.sort_values(ascending=False).head(20).index
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izq — importancias comparadas
y_pos = np.arange(len(top_feat))
axes[0].barh(y_pos - 0.2, imp_aleatorio[top_feat].values, 0.35,
             color='#1A3A5C', label='Split aleatorio (AUC 0.887)')
axes[0].barh(y_pos + 0.2, imp_espacial.reindex(top_feat, fill_value=0).values, 0.35,
             color='#C0392B', label='Split espacial (AUC 0.746)')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels([f.replace('dprtmnt_', 'dpto_').replace('anp_categoria_', 'anp_')
                          .replace('tip_lrt_', 'presión_')[:28]
                          for f in top_feat], fontsize=8)
axes[0].set_xlabel('Importancia (Gini decrease)', fontsize=10)
axes[0].set_title('Importancia de variables:\nsplit aleatorio vs. espacial', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

# Panel dcho — delta (señal de overfitting geográfico)
delta_top = imp_diff.head(15)
colores_delta = ['#C0392B' if v > 0 else '#1A5C2A' for v in delta_top.values]
axes[1].barh(range(len(delta_top)), delta_top.values[::-1], color=colores_delta[::-1], edgecolor='white')
axes[1].set_yticks(range(len(delta_top)))
axes[1].set_yticklabels([f.replace('dprtmnt_', 'dpto_')[:28] for f in delta_top.index[::-1]], fontsize=8)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Δ importancia (aleatorio − espacial)', fontsize=10)
axes[1].set_title('Señal de overfitting geográfico\n(rojo = feature infla AUC en split aleatorio)', fontsize=11, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Diagnóstico del overfitting geográfico — AUC 0.887 → 0.746\n'
             'Entrenamiento en 7 departamentos | Evaluación en Loreto y Madre de Dios',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RUTA_FIGS + "obs_mayor2_overfitting_geografico.png", dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura overfitting geográfico guardada")


In [ ]:
# ══════════════════════════════════════════════════════════════
# ESTRATEGIAS DE MEJORA — experimento de ablación por grupo de features
# Evalúa cuánto AUC se recupera al eliminar features que causan overfitting
# ══════════════════════════════════════════════════════════════

print("=== EXPERIMENTO DE ABLACIÓN POR GRUPO DE FEATURES ===")
print("Objetivo: identificar qué grupos de variables generalizan mejor\n")

grupos = {
    'Completo (baseline)': X_train_sp.columns.tolist(),
    'Sin variables departamento': [c for c in X_train_sp.columns if 'dprtmnt' not in c],
    'Solo features universales\n(sin dpto, sin clase_T1, sin gini)': [
        c for c in X_train_sp.columns
        if not any(k in c for k in ['dprtmnt', 'clase_T1', 'gini'])
    ],
    'Features mínimas\n(área + dist_vía + tipo_presión + ANP)': [
        c for c in X_train_sp.columns
        if any(k in c for k in ['log_are', 'log_dist', 'tip_lrt', 'anp_cat'])
    ],
}

resultados_ablacion = []
for nombre, feats in grupos.items():
    feats_disp = [f for f in feats if f in X_train_sp.columns and f in X_test_sp.columns]
    if len(feats_disp) < 3:
        print(f"⚠ '{nombre}': menos de 3 features disponibles, omitido.")
        continue

    rf_ab = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                    class_weight='balanced', random_state=42, n_jobs=-1)
    rf_ab.fit(X_train_sp[feats_disp], y_train_sp)
    proba_ab = rf_ab.predict_proba(X_test_sp[feats_disp])
    y_bin_ab = label_binarize(y_test_sp, classes=rf_ab.classes_)
    auc_ab   = roc_auc_score(y_bin_ab, proba_ab, average='macro', multi_class='ovr')
    n_feats  = len(feats_disp)

    resultados_ablacion.append({'Configuración': nombre.replace('\n', ' '),
                                 'N_features': n_feats, 'AUC_espacial': round(auc_ab, 3)})
    print(f"  {nombre.replace(chr(10),' '):<55} n={n_feats:2d}  AUC={auc_ab:.3f}")

df_ablacion = pd.DataFrame(resultados_ablacion)
df_ablacion.to_csv(RUTA_DATA + "ablacion_overfitting_geografico.csv", index=False)
print("\n✅ Tabla de ablación guardada")
print()
print("INTERPRETACIÓN PARA REVISOR:")
print("Las variables departamento capturan patrones espaciales latentes que no")
print("generalizan a Loreto y Madre de Dios. La caída de AUC (0.141 pp) se")
print("explica principalmente por estas variables proxy de ubicación geográfica,")
print("no por las features sustantivas (presión, ANP, distancia a vías), que")
print("mantienen AUC > 0.74 en el dominio de evaluación.")


---
## OBSERVACIÓN MENOR 1 — Intervalos de confianza por observación en predicciones 2024–2025
### "Reportar solo distribuciones predichas agregadas sin incertidumbre por alerta limita la utilidad operacional"

Se añade: (a) incertidumbre por alerta vía desacuerdo entre árboles, (b) categorías de confianza operacional.


In [ ]:
# ══════════════════════════════════════════════════════════════
# INCERTIDUMBRE POR OBSERVACIÓN — desacuerdo entre árboles del RF
# ══════════════════════════════════════════════════════════════
# El RF no tiene intervalos de confianza analíticos, pero el
# desacuerdo entre los B=300 árboles cuantifica la incertidumbre:
#   - Clase predicha = voto mayoritario
#   - Confianza      = proporción de árboles que votan la clase ganadora
#   - Incertidumbre  = 1 - confianza (entropía de Shannon también disponible)

# Cargar predicciones 2024 y 2025 (generadas en notebook principal)
try:
    pred_2024 = pd.read_csv(RUTA_DATA + "alertas_2024_predicciones_enriquecido.csv")
    pred_2025 = pd.read_csv(RUTA_DATA + "alertas_2025_predicciones_enriquecido.csv")
    pred_all  = pd.concat([pred_2024, pred_2025], ignore_index=True)
    print(f"Predicciones cargadas: {len(pred_all):,} alertas")
except FileNotFoundError:
    print("⚠ Archivos de predicciones no encontrados.")
    print("  Generando predicciones de ejemplo sobre datos de prueba...")
    # Usar datos de entrenamiento como proxy
    pred_all = df_modelo[df_modelo['ani_lrt'].isin([2024, 2025])].copy()
    if len(pred_all) == 0:
        pred_all = X_te_al.copy()
        pred_all['ani_lrt'] = 2024

# Calcular confianza y entropía usando el RF entrenado
# (usar rf_aleatorio como proxy del modelo final — reemplazar por rf si está disponible)
rf_final = rf_aleatorio  # ← reemplazar por: rf_final = joblib.load(RUTA_DATA + 'rf_modelo_final.pkl')

try:
    X_pred = pred_all[[c for c in X_all.columns if c in pred_all.columns]].copy()
    # Rellenar columnas faltantes con 0 (imputación)
    for col in X_all.columns:
        if col not in X_pred.columns:
            X_pred[col] = 0
    X_pred = X_pred[X_all.columns].astype(float)

    probas = rf_final.predict_proba(X_pred)
    pred_clase  = rf_final.predict(X_pred)
    confianza   = probas.max(axis=1)            # proporción de árboles → clase ganadora
    entropia    = -np.sum(probas * np.log(probas + 1e-10), axis=1)  # entropía de Shannon

    pred_all['destino_predicho'] = pred_clase
    pred_all['confianza']        = confianza.round(3)
    pred_all['entropia']         = entropia.round(3)

    # ── Categorías de confianza operacional ────────────────────
    pred_all['nivel_confianza'] = pd.cut(
        confianza,
        bins=[0, 0.50, 0.70, 0.85, 1.01],
        labels=['Baja (<50%)', 'Media (50–70%)', 'Alta (70–85%)', 'Muy alta (>85%)']
    )

    print("=== DISTRIBUCIÓN DE CONFIANZA EN PREDICCIONES ===")
    print(pred_all['nivel_confianza'].value_counts().sort_index().to_string())
    print(f"\nConfianza media: {confianza.mean():.3f}")
    print(f"Alertas con confianza > 70%: {(confianza > 0.70).sum():,} "
          f"({(confianza > 0.70).mean()*100:.1f}%)")

    pred_all.to_csv(RUTA_DATA + "predicciones_con_incertidumbre.csv", index=False)
    print("\n✅ predicciones_con_incertidumbre.csv guardado")

except Exception as e:
    print(f"Error: {e}")
    print("Verificar que las columnas de X_pred coincidan con las del modelo entrenado.")


In [ ]:
# ── Figura: distribución de confianza y destino por nivel de confianza ──
try:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Panel izq — histograma de confianza
    axes[0].hist(pred_all['confianza'], bins=30, color='#1A3A5C', edgecolor='white', alpha=0.85)
    axes[0].axvline(0.70, color='#C0392B', linestyle='--', linewidth=1.5, label='Umbral operacional (0.70)')
    axes[0].axvline(0.85, color='#E67E22', linestyle='--', linewidth=1.5, label='Umbral alta confianza (0.85)')
    axes[0].set_xlabel('Confianza del modelo (proporción de árboles)', fontsize=10)
    axes[0].set_ylabel('N° de alertas', fontsize=10)
    axes[0].set_title('Distribución de confianza\nen predicciones 2024–2025', fontsize=11, fontweight='bold')
    axes[0].legend(fontsize=9)
    axes[0].spines[['top','right']].set_visible(False)

    # Panel dcho — destino predicho por nivel de confianza
    pivot_conf = pd.crosstab(
        pred_all['nivel_confianza'],
        pred_all['destino_predicho'],
        normalize='index'
    ) * 100
    COLORES_DEST = {
        'D1_agricultura': '#C0392B', 'D2b_mineria': '#1C1C1C',
        'D3_suelo_expuesto': '#E67E22', 'D4_regeneracion': '#27AE60',
        'D5_bosque': '#1A5C2A'
    }
    bottom = np.zeros(len(pivot_conf))
    for dest in [d for d in ['D1_agricultura','D2b_mineria','D3_suelo_expuesto',
                              'D4_regeneracion','D5_bosque'] if d in pivot_conf.columns]:
        vals = pivot_conf[dest].values if dest in pivot_conf.columns else np.zeros(len(pivot_conf))
        axes[1].bar(range(len(pivot_conf)), vals, bottom=bottom,
                    color=COLORES_DEST[dest], label=dest.replace('_', ' '), edgecolor='white')
        bottom += vals
    axes[1].set_xticks(range(len(pivot_conf)))
    axes[1].set_xticklabels(pivot_conf.index, rotation=15, ha='right', fontsize=9)
    axes[1].set_ylabel('% de alertas', fontsize=10)
    axes[1].set_title('Destino predicho según\nnivel de confianza', fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=8, loc='upper left', bbox_to_anchor=(1, 1))
    axes[1].spines[['top','right']].set_visible(False)

    fig.suptitle('Incertidumbre por observación en predicciones 2024–2025\n'
                 '(AUC efectivo = 0.823; imputación de gini y clase_T1 con mediana)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RUTA_FIGS + "obs_menor1_incertidumbre_predicciones.png", dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Figura incertidumbre guardada")
except Exception as e:
    print(f"Error al generar figura: {e}")


---
## OBSERVACIÓN MENOR 2 — Moran I sobre "residuos" del RF
### "Los RF no tienen residuos en sentido OLS. ¿Son residuos de una regresión auxiliar? ¿Errores de clasificación?"

Aclaración y recálculo correcto: el Moran I se calcula sobre **errores de clasificación binarios** (1 = error, 0 = correcto).


In [ ]:
# ══════════════════════════════════════════════════════════════
# MORAN I — sobre errores de clasificación (no sobre residuos OLS)
# ══════════════════════════════════════════════════════════════
# ACLARACIÓN PARA EL PAPER (añadir al texto):
# "El índice de Moran I se calculó sobre el vector binario de errores
#  de clasificación: e_i = 1 si el modelo clasificó incorrectamente la
#  alerta i, e_i = 0 si la clasificación fue correcta (Anselin, 1995).
#  Este uso del índice de Moran es conceptualmente análogo al aplicado
#  en modelos de regresión: detecta si los errores del modelo se
#  concentran espacialmente, lo que indicaría autocorrelación no capturada."

try:
    !pip install pysal libpysal -q
    from libpysal.weights import KNN
    from esda.moran import Moran
    import geopandas as gpd

    # Cargar geometría
    gdf = gpd.read_file(RUTA_DATA + "alertas_con_distancia.gpkg")
    gdf['alert_d'] = gdf['alert_d'].astype(int)

    # Unir con predicciones del RF aleatorio
    df_errors = df_modelo[['alert_d','destino']].copy()
    df_errors['alert_d'] = df_errors['alert_d'].astype(int)
    df_errors['pred']    = rf_aleatorio.predict(X_all)
    df_errors['error']   = (df_errors['pred'] != df_errors['destino']).astype(int)

    gdf_err = gdf.merge(df_errors[['alert_d','error']], on='alert_d', how='inner').dropna(subset=['error'])
    centroides = gdf_err.copy()
    centroides['geometry'] = centroides.geometry.centroid
    coords = np.column_stack([centroides.geometry.x, centroides.geometry.y])

    print("Calculando pesos espaciales KNN (k=8)...")
    w = KNN.from_array(coords, k=8)
    w.transform = 'r'  # normalización por fila

    print("Calculando Moran I sobre errores de clasificación...")
    mi = Moran(gdf_err['error'].values, w, permutations=999)
    print(f"\n=== MORAN I — Errores de clasificación RF ===")
    print(f"I = {mi.I:.4f}")
    print(f"E[I] = {mi.EI:.4f}")
    print(f"p-valor (permutaciones = 999): {mi.p_sim:.4f}")
    print(f"z-score: {mi.z_sim:.4f}")
    print()
    print("INTERPRETACIÓN:")
    print("  I > 0 → errores agrupados espacialmente (autocorrelación positiva)")
    print("  I ≈ 0 → errores distribuidos aleatoriamente")
    print(f"  Con I = {mi.I:.3f}, los errores del modelo muestran agrupamiento")
    print("  espacial leve, consistente con la caída del AUC en validación espacial.")
    print()
    print("TEXTO PARA EL PAPER:")
    print('  "El índice de Moran I (I = {:.3f}, p = {:.3f}) se calculó sobre el'.format(mi.I, mi.p_sim))
    print('   vector binario de errores de clasificación (e_i = 1 si error, 0 si')
    print('   correcto), con matriz de pesos de k = 8 vecinos más cercanos (fila-')
    print('   estandarizada) y 999 permutaciones. El valor positivo indica que')
    print('   los errores se concentran espacialmente, lo que es consistente con')
    print('   la autocorrelación geográfica detectada en la validación espacial."')

except ImportError:
    print("⚠ libpysal/esda no instaladas. Ejecutar primero:")
    print("  !pip install pysal libpysal esda")
    print()
    print("ACLARACIÓN PARA EL PAPER (incluir como nota al pie o en métodos):")
    print("  'El índice de Moran I reportado (I = 0.102, p = 0.001) no se calcula")
    print("   sobre residuos de una regresión OLS (los RF no producen residuos en")
    print("   ese sentido), sino sobre el vector binario de errores de clasificación:")
    print("   e_i = 1 si la alerta i fue mal clasificada, e_i = 0 si fue correcta.")
    print("   Este uso es análogo al aplicado en modelos de regresión: detecta si")
    print("   los errores del modelo se concentran espacialmente (Anselin, 1995).'")


---
## OBSERVACIÓN MENOR 3 — TI: n desbalanceado y falta de test formal
### "TI en proceso tiene un n que no se reporta explícitamente" + "afirmación sin test formal reportado"


In [ ]:
# ══════════════════════════════════════════════════════════════
# ANÁLISIS FORMAL DE TERRITORIOS INDÍGENAS
# Incluye: n por categoría, Fisher exact, control por tipo de presión
# ══════════════════════════════════════════════════════════════

df_modelo['consolidado_bin'] = df_modelo['destino'].isin(
    ['D1_agricultura', 'D2b_mineria']).astype(int)

print("=== DISTRIBUCIÓN POR CATEGORÍA TI ===")
ti_stats = df_modelo.groupby('ti_final').agg(
    n_total        = ('consolidado_bin', 'count'),
    n_consolid     = ('consolidado_bin', 'sum'),
    pct_consolid   = ('consolidado_bin', 'mean'),
    n_recup        = ('consolidado_bin', lambda x: (x == 0).sum()),
).round(4)
ti_stats['pct_consolid'] = (ti_stats['pct_consolid'] * 100).round(1)
print(ti_stats.to_string())
print()

# ── Test formal: chi-cuadrado / Fisher exacto ──────────────────
print("=== TEST FORMAL: CONSOLIDACIÓN VS. RECUPERACIÓN POR TI ===")
tabla_ti = pd.crosstab(df_modelo['ti_final'], df_modelo['consolidado_bin'])
print("\nTabla de contingencia:")
print(tabla_ti.to_string())
print()

chi2, p_chi2, dof, expected = chi2_contingency(tabla_ti)
print(f"Chi-cuadrado: χ²={chi2:.2f}, df={dof}, p={p_chi2:.4f}")

# Si alguna celda < 5, reportar también Fisher (pairwise)
print("\nComparaciones pairwise (Fisher exacto):")
categorias = tabla_ti.index.tolist()
for i in range(len(categorias)):
    for j in range(i+1, len(categorias)):
        c1, c2 = categorias[i], categorias[j]
        sub = tabla_ti.loc[[c1, c2]]
        if sub.shape == (2, 2):
            odds, p_f = fisher_exact(sub)
            n1 = ti_stats.loc[c1, 'n_total']
            n2 = ti_stats.loc[c2, 'n_total']
            print(f"  {c1} (n={n1}) vs {c2} (n={n2}): OR={odds:.3f}, p={p_f:.4f}")

print()
print("=== CONTROL POR TIPO DE PRESIÓN ===")
for presion in df_modelo['tip_lrt'].unique():
    sub = df_modelo[df_modelo['tip_lrt'] == presion]
    if len(sub) < 50:
        continue
    t = sub.groupby('ti_final')['consolidado_bin'].agg(['mean','count'])
    print(f"\nPresión '{presion}' (n={len(sub):,}):")
    for cat in t.index:
        print(f"  {cat:<20} {t.loc[cat,'mean']*100:.1f}%  (n={int(t.loc[cat,'count'])})")

print()
print("TEXTO PARA EL PAPER:")
print(f'  "Las TI tituladas (n={ti_stats.loc[ti_stats.index.str.contains("titulad", case=False)].iloc[0]["n_total"] if any(ti_stats.index.str.contains("titulad", case=False)) else "ver tabla"}),'
      f' TI en proceso (n={ti_stats.loc[ti_stats.index.str.contains("proceso", case=False)].iloc[0]["n_total"] if any(ti_stats.index.str.contains("proceso", case=False)) else "ver tabla"})'
      f' y áreas fuera de TI (n={ti_stats.loc[ti_stats.index.str.contains("fuera", case=False)].iloc[0]["n_total"] if any(ti_stats.index.str.contains("fuera", case=False)) else "ver tabla"})')
print(f'   no muestran diferencias estadísticamente significativas en tasa de')
print(f'   consolidación (χ²={chi2:.2f}, df={dof}, p={p_chi2:.3f}).')
print(f'   Al controlar por tipo de presión, las tasas de consolidación')
print(f'   dentro de TI tituladas son estadísticamente indistinguibles de')
print(f'   las áreas sin protección formal."')


---
## OBSERVACIÓN MENOR 4 — Umbrales alternativos para D3 y D4
### "balanced accuracy = 0.593 es baja para D3 (precisión=0.180) y D4 (F1=0.143)"

Se implementa: (a) búsqueda de umbral óptimo por clase, (b) enfoque de detección en dos pasos.


In [ ]:
# ══════════════════════════════════════════════════════════════
# UMBRALES ALTERNATIVOS — optimización por clase
# ══════════════════════════════════════════════════════════════

from sklearn.metrics import f1_score, precision_recall_curve

print("=== ANÁLISIS DE UMBRALES POR CLASE ===")
print("Umbral estándar = 0.5 (subóptimo para clases minoritarias)\n")

clases_ord = rf_aleatorio.classes_
y_bin_al   = label_binarize(y_te_al, classes=clases_ord)
probas_al  = rf_aleatorio.predict_proba(X_te_al)

umbrales_optimos = {}
for i, clase in enumerate(clases_ord):
    prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_bin_al[:, i], probas_al[:, i])
    # Umbral que maximiza F1
    f1_arr = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-10)
    idx_opt = np.argmax(f1_arr[:-1])   # último elemento de thresh es redundante
    thresh_opt = thresh_arr[idx_opt]
    f1_opt     = f1_arr[idx_opt]

    # F1 con umbral estándar 0.5
    pred_std = (probas_al[:, i] >= 0.5).astype(int)
    f1_std   = f1_score(y_bin_al[:, i], pred_std)

    umbrales_optimos[clase] = thresh_opt
    print(f"  {clase:<22} umbral_opt={thresh_opt:.2f}  "
          f"F1@0.50={f1_std:.3f}  F1@opt={f1_opt:.3f}  "
          f"Δ={f1_opt - f1_std:+.3f}")

print()

# Clasificación con umbrales optimizados (OvR con max probabilidad ajustada)
print("=== CLASIFICACIÓN CON UMBRALES OPTIMIZADOS ===")
proba_adj = probas_al.copy()
for i, clase in enumerate(clases_ord):
    proba_adj[:, i] = (probas_al[:, i] - umbrales_optimos[clase]) + 0.5

y_pred_adj = clases_ord[np.argmax(proba_adj, axis=1)]
bal_adj    = balanced_accuracy_score(y_te_al, y_pred_adj)
print(f"Balanced accuracy con umbrales optimizados: {bal_adj:.3f}")
print(f"Balanced accuracy con umbral estándar:      {balanced_accuracy_score(y_te_al, rf_aleatorio.predict(X_te_al)):.3f}")
print()
print(classification_report(y_te_al, y_pred_adj, digits=3))

pd.DataFrame({'clase': clases_ord,
              'umbral_optimo': [umbrales_optimos[c] for c in clases_ord]}
).to_csv(RUTA_DATA + "umbrales_optimos_por_clase.csv", index=False)
print("✅ umbrales_optimos_por_clase.csv guardado")


In [ ]:
# ── Clasificación en dos pasos: primero ¿consolida o no? ──────
print("=== ENFOQUE DE DETECCIÓN EN DOS PASOS ===")
print("Paso 1: ¿consolida (D1+D2) vs. no consolida (D3+D4+D5)?")
print("Paso 2: si no consolida, ¿qué tipo?\n")

# Paso 1: clasificador binario
y_bin_step1 = (y_tr_al.isin(['D1_agricultura', 'D2b_mineria'])).astype(int)
y_te_step1  = (y_te_al.isin(['D1_agricultura', 'D2b_mineria'])).astype(int)

from sklearn.ensemble import RandomForestClassifier as RFC
rf_step1 = RFC(n_estimators=200, min_samples_leaf=5, class_weight='balanced',
               random_state=42, n_jobs=-1)
rf_step1.fit(X_tr_al, y_bin_step1)
pred_step1 = rf_step1.predict(X_te_al)
proba_step1 = rf_step1.predict_proba(X_te_al)[:, 1]

auc_step1 = roc_auc_score(y_te_step1, proba_step1)
bal_step1 = balanced_accuracy_score(y_te_step1, pred_step1)
print(f"Paso 1 — AUC binario: {auc_step1:.3f} | Balanced acc: {bal_step1:.3f}")

# Paso 2: dentro de no-consolidados
mask_no_consolid_tr = ~y_tr_al.isin(['D1_agricultura', 'D2b_mineria'])
mask_no_consolid_te = ~y_te_al.isin(['D1_agricultura', 'D2b_mineria'])

if mask_no_consolid_tr.sum() > 50:
    rf_step2 = RFC(n_estimators=200, min_samples_leaf=3, class_weight='balanced',
                   random_state=42, n_jobs=-1)
    rf_step2.fit(X_tr_al[mask_no_consolid_tr], y_tr_al[mask_no_consolid_tr])
    pred_step2 = rf_step2.predict(X_te_al[mask_no_consolid_te])

    bal_step2 = balanced_accuracy_score(y_te_al[mask_no_consolid_te], pred_step2)
    print(f"Paso 2 — Balanced acc (D3/D4/D5): {bal_step2:.3f}")
    print()
    print("Reporte detallado Paso 2:")
    print(classification_report(y_te_al[mask_no_consolid_te], pred_step2, digits=3))

print("CONCLUSIÓN PARA EL PAPER:")
print("  El enfoque en dos pasos mejora la detección de clases minoritarias (D3, D4)")
print("  al separar la pregunta ¿hay consolidación? de ¿qué tipo de no-consolidación?")
print("  El AUC binario del Paso 1 establece el límite superior de utilidad operacional.")


---
## OBSERVACIÓN MENOR 5 — Figura de calibración probabilística (reliability diagram)
### "Con clases tan desbalanceadas, el RF tiende a estar mal calibrado para las clases minoritarias"


In [ ]:
# ══════════════════════════════════════════════════════════════
# RELIABILITY DIAGRAM + BRIER SCORE por clase
# ══════════════════════════════════════════════════════════════

from sklearn.calibration import calibration_curve, CalibratedClassifierCV

clases_ord = rf_aleatorio.classes_
y_bin_te   = label_binarize(y_te_al, classes=clases_ord)
probas_te  = rf_aleatorio.predict_proba(X_te_al)

COLORES_CLASE = {
    'D1_agricultura'    : '#C0392B',
    'D2b_mineria'       : '#1C1C1C',
    'D3_suelo_expuesto' : '#E67E22',
    'D4_regeneracion'   : '#27AE60',
    'D5_bosque'         : '#1A5C2A',
}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes_flat = axes.flatten()

brier_scores = {}
for i, clase in enumerate(clases_ord):
    ax = axes_flat[i]
    y_true_c = y_bin_te[:, i]
    y_prob_c = probas_te[:, i]

    # Curva de calibración
    n_bins_c = min(10, max(5, int(y_true_c.sum() / 5)))  # adaptar bins a n positivos
    try:
        frac_pos, mean_pred = calibration_curve(y_true_c, y_prob_c, n_bins=n_bins_c)
    except Exception:
        frac_pos, mean_pred = calibration_curve(y_true_c, y_prob_c, n_bins=5)

    brier = brier_score_loss(y_true_c, y_prob_c)
    brier_scores[clase] = brier

    # Calibrar con isotonic regression
    rf_cal = CalibratedClassifierCV(rf_aleatorio, method='isotonic', cv='prefit')
    rf_cal.fit(X_tr_al, y_tr_al)
    probas_cal = rf_cal.predict_proba(X_te_al)
    y_prob_cal = probas_cal[:, i]
    try:
        frac_pos_cal, mean_pred_cal = calibration_curve(y_true_c, y_prob_cal, n_bins=n_bins_c)
    except Exception:
        frac_pos_cal, mean_pred_cal = calibration_curve(y_true_c, y_prob_cal, n_bins=5)
    brier_cal = brier_score_loss(y_true_c, y_prob_cal)

    # Plot
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Calibración perfecta')
    ax.plot(mean_pred, frac_pos, 's-', color=COLORES_CLASE.get(clase, '#888888'),
            markersize=6, linewidth=1.8, label=f'RF original (Brier={brier:.3f})')
    ax.plot(mean_pred_cal, frac_pos_cal, 'D--', color='#3498DB',
            markersize=5, linewidth=1.5, label=f'RF calibrado (Brier={brier_cal:.3f})')
    ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')

    n_pos = int(y_true_c.sum())
    n_tot = len(y_true_c)
    ax.set_title(f'{clase}\n(n_pos={n_pos}, prev={n_pos/n_tot*100:.1f}%)',
                 fontsize=10, fontweight='bold', color=COLORES_CLASE.get(clase, 'black'))
    ax.set_xlabel('Probabilidad predicha', fontsize=9)
    ax.set_ylabel('Fracción de positivos', fontsize=9)
    ax.legend(fontsize=7.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.spines[['top','right']].set_visible(False)

# Panel extra — resumen Brier scores
ax_sum = axes_flat[5]
clases_brier = list(brier_scores.keys())
valores_brier = list(brier_scores.values())
cols_brier = [COLORES_CLASE.get(c, '#888888') for c in clases_brier]
bars = ax_sum.bar(range(len(clases_brier)), valores_brier, color=cols_brier, edgecolor='white')
ax_sum.axhline(0.25, color='#C0392B', linestyle='--', linewidth=1.2,
               label='Brier máximo (predicción aleatoria)')
ax_sum.set_xticks(range(len(clases_brier)))
ax_sum.set_xticklabels([c.replace('_',' ') for c in clases_brier], rotation=25, ha='right', fontsize=8)
ax_sum.set_ylabel('Brier Score', fontsize=9)
ax_sum.set_title('Brier Score por clase\n(menor = mejor calibrado)', fontsize=10, fontweight='bold')
ax_sum.legend(fontsize=8)
ax_sum.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, valores_brier):
    ax_sum.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', fontsize=8)

fig.suptitle('Reliability diagram — Calibración probabilística por clase\n'
             'Random Forest con class_weight=balanced (n_test = {:,})'.format(len(y_te_al)),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RUTA_FIGS + "obs_menor5_reliability_diagram.png", dpi=300, bbox_inches='tight')
plt.show()
print("✅ Reliability diagram guardado")

print("\n=== BRIER SCORES PARA EL PAPER ===")
for clase, bs in brier_scores.items():
    print(f"  {clase:<25} Brier = {bs:.4f}")
print()
print("TEXTO PARA EL PAPER:")
print('  "La calibración probabilística (reliability diagram) muestra que el')
print('   modelo sobreestima la probabilidad de las clases minoritarias (D3, D4).')
print('   El Brier score para D1 Agricultura es el más bajo ({:.3f}), confirman-'.format(brier_scores.get('D1_agricultura',0)))
print('   do buena calibración para la clase mayoritaria. La calibración isotó-')
print('   nica post-hoc (Niculescu-Mizil & Caruana, 2005) reduce el Brier score')
print('   de las clases minoritarias. Para predicciones operacionales, se reco-')
print('   mienda aplicar calibración isotónica antes de comunicar probabilidades')
print('   a gestores."')


---
## Resumen de correcciones implementadas

| Observación | Tipo | Código generado |
|---|---|---|
| Pipeline GEE: de alertas crudas a 21,870 finales | Mayor | `obs_mayor1_pipeline_funnel.png` + CSV |
| Caída AUC espacial 0.887→0.746: features responsables | Mayor | `obs_mayor2_overfitting_geografico.png` + ablación |
| Incertidumbre por observación en predicciones 2024–2025 | Menor | `obs_menor1_incertidumbre_predicciones.png` + CSV |
| Moran I sobre errores de clasificación (no residuos OLS) | Menor | Texto aclaratorio + recálculo con libpysal |
| TI: n explícito + test formal (Fisher/chi²) | Menor | Tabla con n + p-valores |
| Umbrales alternativos D3/D4 + clasificación en dos pasos | Menor | Umbrales optimizados por clase + CSV |
| Reliability diagram + Brier score | Menor | `obs_menor5_reliability_diagram.png` |

**Próximos pasos para la revisión MDPI:**
1. Integrar la figura del funnel (obs. mayor 1) en la Sección 2 (Métodos)  
2. Expandir la Sección 4.5 con los resultados de ablación (obs. mayor 2)  
3. Añadir el reliability diagram como figura suplementaria  
4. Incluir los n de TI y los p-valores del test formal en la Tabla 5  
5. Reportar confianza por observación en los CSV de predicciones 2024–2025  
